# Lab 14: Hate Speech and Offensive Language Detection

**Content warning:** this dataset consists of real tweets, many containing slurs and abusive language. The notebook masks offensive words when it displays terms and examples.

**Task:** classify tweets as hate speech, offensive language, or neither.

**Dataset:** Hate Speech and Offensive Language (Davidson, Warmsley, Macy and Weber, 2017: https://github.com/t-davidson/hate-speech-and-offensive-language), included as `data/labeled_data.csv` as downloaded: 24,783 tweets, each labelled by at least three crowd workers. `class` is the majority label: 0 = hate speech, 1 = offensive language, 2 = neither. The columns `hate_speech`, `offensive_language` and `neither` count how many annotators chose each label.

**What we will do:**
1. Load the tweets and look at the classes.
2. Preprocess: remove URLs and mentions, turn hashtags into words, remove stopwords, stem.
3. Represent the tweets with TF-IDF: unigrams only vs unigrams + bigrams.
4. Train multiclass logistic regression and multinomial Naive Bayes.
5. Evaluate with macro-F1 and the confusion matrix, focusing on hate vs offensive.
6. Report the top terms per class and discuss what errors mean in practice.

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 150)
pd.set_option("display.max_colwidth", 120)
tweets = pd.read_csv("data/labeled_data.csv", index_col=0)
class_names = {0: "hate", 1: "offensive", 2: "neither"}
tweets["label"] = tweets["class"].map(class_names)
print(tweets.shape)
print(tweets["label"].value_counts().to_dict())
print(tweets["label"].value_counts(normalize=True).round(3).to_dict())

The classes are very **imbalanced**: 77% offensive, 17% neither, only 6% hate speech. Accuracy would be dominated by the offensive class; **macro-F1** (the F1 score computed for each class separately, then averaged with equal weight) treats the small hate class as important as the large one.

How much did the annotators agree?

In [ ]:
tweets["agreement"] = tweets[["hate_speech", "offensive_language", "neither"]].max(axis=1) / tweets["count"]
print(tweets.groupby("label")["agreement"].mean().round(3).rename("average share of annotators agreeing with the label"))

Annotators agree much less on hate speech than on the other classes: the line between "hate" and "offensive" is genuinely hard to draw, even for people. A model can hardly be expected to do better than the people who labelled the data.

To display tweets and terms without printing slurs in full, we mask words from a small list of the most offensive terms that appear in this data (first letter kept, the rest replaced by `*`). The models still see the original words.

In [ ]:
mask_pattern = re.compile(r"\b(n[i1]gg\w*|f[a4]gg?\w*|b[i1]tch\w*|h[o0]e?s?\b|c[u]nt\w*|r[e]tard\w*|wh[o0]re\w*|sl[u]t\w*|k[i1]ke\w*|sp[i1]c\w*|ch[i1]nk\w*|d[y]ke\w*|qu[e]er\w*|tw[a]t\w*|p[u]ssy\w*|f[u]ck\w*|sh[i1]t\w*|coon\w*|wetback\w*|beaner\w*|raghead\w*|trann\w*|towelhead\w*|motherf\w*|pussi\w*)", re.I)
def mask(text):
    return mask_pattern.sub(lambda m: m.group(0)[0] + "*" * (len(m.group(0)) - 1), str(text))

for label in ["hate", "offensive", "neither"]:
    print(label.upper(), "|", mask(tweets.loc[tweets["label"] == label, "tweet"].iloc[2])[:140])

## 2. Preprocessing

Tweets contain a lot of noise:
- **URLs** (`http://t.co/...`) and **mentions** (`@username`): mentions identify people, not content; we remove both.
- **Retweet markers** (`RT`) and HTML entities (`&amp;`, `&#128514;` for emoji).
- **Hashtags**: `#whitepower` carries meaning, so we keep the word and drop the `#` (splitting joined words like "whitepower" would need more work).
- Then lowercase, keep letters only, remove stopwords, and stem with the Porter stemmer.

In [ ]:
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

stemmer = PorterStemmer()
def preprocess(text):
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)      # URLs
    text = re.sub(r"@\w+", " ", text)                       # mentions
    text = re.sub(r"&#?\w+;", " ", text)                    # HTML entities and emoji codes
    text = re.sub(r"\bRT\b", " ", text)                     # retweet marker
    text = text.replace("#", " ")                           # hashtags -> words
    text = re.sub(r"[^a-z\s]", " ", text.lower())
    tokens = [stemmer.stem(t) for t in text.split() if t not in ENGLISH_STOP_WORDS and len(t) > 1]
    return " ".join(tokens)

tweets["clean"] = tweets["tweet"].map(preprocess)
for raw, clean in tweets[["tweet", "clean"]].head(3).values:
    print(mask(raw)[:110], "\n   ->", mask(clean)[:110])

## 3. TF-IDF: unigrams vs unigrams + bigrams

TF-IDF gives each term a weight: its frequency in the tweet times its inverse document frequency (rare terms count more), scaled to length 1. With **bigrams** (pairs of consecutive words) the model can tell "white trash" or "go back" from the words used alone, at the cost of many more features.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import f1_score, classification_report, confusion_matrix

x_train, x_test, y_train, y_test = train_test_split(tweets["clean"], tweets["label"], test_size=0.25,
                                                    stratify=tweets["label"], random_state=0)
labels = ["hate", "offensive", "neither"]

## 4. Models

- **Multiclass logistic regression** (softmax): one weight vector per class, probabilities from the softmax, trained with cross-entropy. `class_weight="balanced"` makes the rare hate class count as much as the others in the loss.
- **Multinomial Naive Bayes**: Bayes' theorem with word counts assumed independent within each class.

In [ ]:
results, fitted = [], {}
for ngrams in [(1, 1), (1, 2)]:
    for name, model in [("logistic regression", LogisticRegression(C=3, class_weight="balanced", max_iter=3000)),
                        ("naive bayes", MultinomialNB(alpha=0.1))]:
        pipe = Pipeline([("tfidf", TfidfVectorizer(ngram_range=ngrams, min_df=2, sublinear_tf=True)), ("model", model)])
        pipe.fit(x_train, y_train)
        pred = pipe.predict(x_test)
        key = f"{name}, {'unigrams' if ngrams == (1, 1) else 'uni + bigrams'}"
        fitted[key] = (pipe, pred)
        per_class = f1_score(y_test, pred, labels=labels, average=None)
        results.append({"model": key, "features": len(pipe[0].vocabulary_), "macro F1": f1_score(y_test, pred, average="macro"),
                        **{f"F1 {l}": v for l, v in zip(labels, per_class)}})
pd.DataFrame(results).round(3)

## 5. Evaluation

In [ ]:
best = max(results, key=lambda r: r["macro F1"])["model"]
pipe, pred = fitted[best]
print("best by macro-F1:", best)
print(classification_report(y_test, pred, labels=labels, digits=3))
cm = pd.DataFrame(confusion_matrix(y_test, pred, labels=labels), index=[f"actual {l}" for l in labels], columns=[f"pred {l}" for l in labels])
cm

### Hate vs offensive confusion
The confusion matrix shows where the errors are: most hate tweets that are missed are classified as **offensive**, not as neither. The two classes share most of their vocabulary (slurs and insults appear in both); what distinguishes hate speech is often *who* is targeted and *why* (a group, because of race, religion, sexuality), which a bag of words captures only partly.

In [ ]:
test_view = pd.DataFrame({"tweet": tweets.loc[x_test.index, "tweet"], "actual": y_test, "predicted": pred})
print("HATE tweets predicted as offensive:")
for t in test_view[(test_view.actual == "hate") & (test_view.predicted == "offensive")]["tweet"].head(5):
    print("  -", mask(t)[:140])
print("\nOFFENSIVE tweets predicted as hate:")
for t in test_view[(test_view.actual == "offensive") & (test_view.predicted == "hate")]["tweet"].head(5):
    print("  -", mask(t)[:140])

## 6. Deliverables

### Top terms per class (logistic regression weights)
For each class, the terms with the largest weights in that class's weight vector (masked where offensive).

In [ ]:
lr_pipe = fitted["logistic regression, uni + bigrams"][0]
vocab = lr_pipe[0].get_feature_names_out()
model = lr_pipe[-1]
for i, cls in enumerate(model.classes_):
    top = np.argsort(model.coef_[i])[::-1][:20]
    print(f"{cls.upper():10s}:", ", ".join(mask(vocab[j]) for j in top))

The **hate** class is dominated by slurs targeting race, ethnicity and sexuality, and by words naming groups ("white", "jew"). The **offensive** class has general profanity and sexist insults, used very often in casual, non-hateful ways in this data.

The **neither** class is revealing: its top words (yankees, charlie, brownies, oreo, bird, zebra, twinkie, crow) are words that *can* be slurs but are mostly used innocently: sports teams, food, animals. The tweets were collected by searching Twitter for terms from a hate-speech lexicon, so every class is full of these words; the model learned which ones are usually harmless. A model trained this way is tied to that lexicon and that period's slang.

Note also that identity words such as "white" and "jew" push a tweet towards "hate" even when the tweet is not hateful at all, a known bias of such models (see below).

### Ethics note: what the errors mean

- **False positives** (an innocent tweet flagged as hate or offensive): on a platform this means removing posts or suspending users who did nothing wrong. Such errors are not spread evenly: models trained on data like this one flag tweets written in African-American English, or tweets that merely *mention* a group or *quote* a slur to criticise it, far more often than others (studies of this very dataset found exactly that). Errors then fall on the communities the system was meant to protect.
- **False negatives** (hate speech that is missed): harmful content stays up, and its targets bear the cost. Coded language, new slurs and sarcasm are the typical misses, and they change quickly to evade filters.
- **The labels themselves are judgements.** Annotators disagreed most on the hate class; the crowd workers' backgrounds shape what counts as "hate" or "offensive". A model trained on these labels inherits those choices and presents them as objective.
- **In practice**, such models should support human moderators (ranking what to review) rather than take automatic action, be evaluated separately on different groups and dialects, and be retrained as language changes. Macro-F1 around the values above means many individual decisions would be wrong.